# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Vedika1304-05/flyrank-internship-ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

THE RULE:

Score every page primarily by how badly it under-clicks for the visibility it already has, with traffic size as a secondary tiebreaker and deliberately leave staleness out, since we proved it doesn't predict anything here.
A page earns a high score when it's actually being seen (real impressions) but isn't converting that visibility into clicks because that's the single signal we found genuinely separates pages that go on to decline from pages that don't (59% vs. 36% decline rate, our strongest finding). Among pages with similarly weak CTR, we then nudge higher-traffic pages further up the queue, since bigger pages losing ground matters more to catch than small ones.

FORMULA: rule_score = 0.75 * ctr_weakness_score + 0.25 * volume_score
REASON CODES:
1. weak_ctr_visible_page -> CTR is in the bottom 25% (Q1, decline rate 59.3%) and the page has meaningful traffic (≥100 impressions, to avoid the zero-inflation noise you already found).
2. high_traffic_at_risk -> CTR is not in the bottom quartile, but the page sits in the top 25% by volume — flagged mainly because a large amount of traffic is on the line, even without a severe CTR problem.
3. moderate_opportunity -> Neither condition above is severe, but the combined rule_score still places the page in the reviewable range — a catch-all for pages worth a lower-priority look.

ACTIONABLE LABELS:
weak_ctr_visible_page-> review_metadata
high_traffic_at_risk-> monitor_closely
moderate_opportunity -> monitor (low priority)





In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [14]:
import os, getpass
import duckdb
import pandas as pd

# ============================================================
# 1. Authenticate to Hugging Face
# ============================================================
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients': f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content': f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':  f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
}

# ============================================================
# 2. Define the windows
# ============================================================
MONTH_START     = "2026-01-01"
DECISION_POINT  = "2026-03-31"   # "pretend today" — features use only rows <= this date
LABEL_START     = "2026-04-01"   # label window starts strictly AFTER the decision point
LABEL_END       = "2026-05-01"
DECLINE_THRESHOLD = -0.20        # a 20%+ drop counts as "declining"

# ============================================================
# 3. Build FEATURES from March (on or before the decision point)
# ============================================================
feature_frame = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS total_impressions_30d,
        SUM(gsc_clicks) AS total_clicks_30d,
        ROUND(SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0), 4) AS ctr_30d,
        ROUND(AVG(gsc_avg_position), 2) AS avg_position_30d,
        COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END) AS days_with_impressions_30d
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '{MONTH_START}'
      AND report_date <= DATE '{DECISION_POINT}'
    GROUP BY client_hash_id, content_hash_id
    HAVING SUM(gsc_impressions) > 0
""").df()
print(f"feature_frame: {feature_frame.shape[0]:,} rows")

# ============================================================
# 4. Build the LABEL from April (strictly after the decision point)
# ============================================================
label_frame = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions_next30d
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '{LABEL_START}' AND report_date < DATE '{LABEL_END}'
    GROUP BY client_hash_id, content_hash_id
""").df()
print(f"label_frame: {label_frame.shape[0]:,} rows")

# ============================================================
# 5. Merge features + label, compute is_declining
# ============================================================
dataset = feature_frame.merge(label_frame, on=["client_hash_id", "content_hash_id"], how="inner")

dataset["pct_change"] = (
    (dataset["impressions_next30d"] - dataset["total_impressions_30d"])
    / dataset["total_impressions_30d"].replace(0, 1)
)
dataset["is_declining"] = (dataset["pct_change"] <= DECLINE_THRESHOLD).astype(int)
print(f"dataset (features + label merged): {dataset.shape[0]:,} rows")
print(f"Decline rate: {dataset['is_declining'].mean():.3f}")

# ============================================================
# 6. Join in content age / staleness from dim_content
# (real column names: content_created_date, content_updated_date)
# ============================================================
content_age = con.sql(f"""
    SELECT
        content_hash_id,
        DATE_DIFF('day', content_updated_date, DATE '{DECISION_POINT}') AS days_since_last_update,
        DATE_DIFF('day', content_created_date, DATE '{DECISION_POINT}') AS content_age_days
    FROM {TABLES['dim_content']}
""").df()

dataset = dataset.merge(
    content_age[["content_hash_id", "days_since_last_update", "content_age_days"]],
    on="content_hash_id", how="left"
)

# ============================================================
# 7. Verification — prove the windows don't overlap
# ============================================================
window_check = con.sql(f"""
    SELECT
        (SELECT MAX(report_date) FROM {TABLES['fact_daily']}
         WHERE report_date <= DATE '{DECISION_POINT}') AS last_feature_date,
        (SELECT MIN(report_date) FROM {TABLES['fact_daily']}
         WHERE report_date >= DATE '{LABEL_START}') AS first_label_date
""").df()
no_overlap = window_check['last_feature_date'][0] < window_check['first_label_date'][0]
print(f"\nWindows don't overlap: {no_overlap}")

print(f"\nFinal dataset shape: {dataset.shape}")
print(dataset.columns.tolist())
dataset.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

feature_frame: 203,074 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

label_frame: 362,172 rows
dataset (features + label merged): 194,010 rows
Decline rate: 0.769


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Windows don't overlap: True

Final dataset shape: (194010, 12)
['client_hash_id', 'content_hash_id', 'total_impressions_30d', 'total_clicks_30d', 'ctr_30d', 'avg_position_30d', 'days_with_impressions_30d', 'impressions_next30d', 'pct_change', 'is_declining', 'days_since_last_update', 'content_age_days']


,client_hash_id,content_hash_id,total_impressions_30d,total_clicks_30d,ctr_30d,avg_position_30d,days_with_impressions_30d,impressions_next30d,pct_change,is_declining,days_since_last_update,content_age_days
0,client_62f4a7e64f5e0096,content_65b8a610174a1036,6017.0,48.0,0.0080,5.07,90,1931.0,-0.679076,1,-95,195
1,client_62f4a7e64f5e0096,content_80071808216aef39,124246.0,288.0,0.0023,4.05,90,39059.0,-0.685632,1,-95,195
2,client_62f4a7e64f5e0096,content_9a136f9ba3924c74,60387.0,240.0,0.0040,3.37,90,18363.0,-0.695911,1,-95,195
3,client_62f4a7e64f5e0096,content_200d6a6d1cac69b7,7495.0,3.0,0.0004,0.74,90,1495.0,-0.800534,1,-95,195
4,client_62f4a7e64f5e0096,content_0d440941b1c6807f,7660.0,10.0,0.0013,3.07,90,1196.0,-0.843864,1,-95,195


In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import pandas as pd
import numpy as np

scored = dataset.copy()
scored = scored[scored["total_impressions_30d"] >= 100].copy()

# ============================================================
# Position tiers
# ============================================================
scored["position_tier"] = pd.cut(
    scored["avg_position_30d"],
    bins=[0, 3, 10, 20, 50, 1000],
    labels=["1-3", "4-10", "11-20", "21-50", "51+"]
)

# ============================================================
# FIXED: expected CTR baseline — median, computed only from pages
# with real engagement (ctr_30d > 0), so the zero-CTR pile doesn't
# drag down the very benchmark it's being compared against.
# ============================================================
healthy_pages = scored[scored["ctr_30d"] > 0]
tier_expected_ctr = healthy_pages.groupby("position_tier", observed=True)["ctr_30d"].median()
scored["expected_ctr_for_tier"] = scored["position_tier"].map(tier_expected_ctr).astype(float)

print("Expected CTR by position tier (healthy pages only, median):")
print(tier_expected_ctr.round(4))

scored["ctr_gap"] = (scored["expected_ctr_for_tier"] - scored["ctr_30d"]).clip(lower=0)

# ============================================================
# The rule
# ============================================================
scored["ctr_weakness_score"] = scored["ctr_gap"].rank(pct=True)
scored["volume_score"] = scored["total_impressions_30d"].rank(pct=True)

scored["rule_score"] = (
    0.75 * scored["ctr_weakness_score"] +
    0.25 * scored["volume_score"]
)

# ============================================================
# One reason code per page
# ============================================================
CTR_GAP_THRESHOLD = scored["ctr_gap"].quantile(0.75)
VOLUME_Q3_THRESHOLD = scored["total_impressions_30d"].quantile(0.75)

def assign_reason_code(row):
    if row["ctr_gap"] >= CTR_GAP_THRESHOLD:
        return "weak_ctr_visible_page"
    elif row["total_impressions_30d"] >= VOLUME_Q3_THRESHOLD:
        return "high_traffic_at_risk"
    else:
        return "moderate_opportunity"

scored["reason_code"] = scored.apply(assign_reason_code, axis=1)

ACTION_MAP = {
    "weak_ctr_visible_page": "review_metadata",
    "high_traffic_at_risk":  "monitor_closely",
    "moderate_opportunity":  "monitor",
}
scored["suggested_action"] = scored["reason_code"].map(ACTION_MAP)

# ============================================================
# Rank and write
# ============================================================
scored = scored.sort_values("rule_score", ascending=False).reset_index(drop=True)
scored["rank"] = scored.index + 1

output_cols = [
    "rank", "content_hash_id", "client_hash_id", "rule_score",
    "reason_code", "suggested_action",
    "position_tier", "ctr_30d", "expected_ctr_for_tier", "ctr_gap",
    "total_impressions_30d", "avg_position_30d", "is_declining"
]

scored[output_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)

print("Rewritten. Reason codes in the fresh CSV:")
print(pd.read_csv("work/outputs/baseline_action_score.csv")["reason_code"].value_counts())

Expected CTR by position tier (healthy pages only, median):
position_tier
1-3      0.0028
4-10     0.0026
11-20    0.0024
21-50    0.0018
51+      0.0028
Name: ctr_30d, dtype: float64
Rewritten. Reason codes in the fresh CSV:
reason_code
moderate_opportunity     58621
weak_ctr_visible_page    29888
high_traffic_at_risk     27021
Name: count, dtype: int64


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20 = pd.read_csv("work/outputs/baseline_action_score.csv").head(20)

CAVEAT_TEMPLATES = {
    "weak_ctr_visible_page": (
        "wrong if the CTR gap reflects a genuine search-intent mismatch rather than "
        "a fixable title/meta problem — e.g. the page ranks well for a query it wasn't "
        "really written to satisfy, so no amount of metadata rewriting recovers clicks."
    ),
    "high_traffic_at_risk": (
        "wrong if this page's high volume is simply a large, stable performer rather "
        "than an actual weakness — flagged mainly by size, without a real CTR problem "
        "backing it up."
    ),
    "moderate_opportunity": (
        "wrong if this page has no real problem and just crossed the combined score "
        "threshold by averaging two mediocre-but-fine numbers — this bucket is the "
        "least specific, so false positives are most likely here."
    ),
}

def confidence_note(row, full_df):
    if row["total_impressions_30d"] >= full_df["total_impressions_30d"].quantile(0.75):
        conf = "HIGH"
        why = "well above median traffic — CTR gap is measured against a lot of real data"
    elif row["total_impressions_30d"] >= full_df["total_impressions_30d"].median():
        conf = "MEDIUM"
        why = "moderate traffic — reasonably reliable, but less data-backed than top-quartile pages"
    else:
        conf = "LOW"
        why = "lower traffic — CTR/gap estimate is noisier at this volume"
    return f"{conf} — {why}"

full_df = pd.read_csv("work/outputs/baseline_action_score.csv")

print("="*100)
print("TOP 20 REVIEW")
print("="*100)
for _, row in top20.iterrows():
    conf = confidence_note(row, full_df)
    caveat = CAVEAT_TEMPLATES[row["reason_code"]]
    print(f"\n#{row['rank']}  {row['content_hash_id']}  "
          f"score={row['rule_score']:.3f}  tier={row['position_tier']}  "
          f"ctr_gap={row['ctr_gap']:.5f}  is_declining={row['is_declining']}")
    print(f"  Action:          {row['suggested_action']}")
    print(f"  Reason code:     {row['reason_code']}")
    print(f"  Confidence:      {conf}")
    print(f"  Would be wrong:  {caveat}")

TOP 20 REVIEW

#1  content_b9acd1ebff7d34ff  score=0.960  tier=1-3  ctr_gap=0.00270  is_declining=0
  Action:          review_metadata
  Reason code:     weak_ctr_visible_page
  Confidence:      HIGH — well above median traffic — CTR gap is measured against a lot of real data
  Would be wrong:  wrong if the CTR gap reflects a genuine search-intent mismatch rather than a fixable title/meta problem — e.g. the page ranks well for a query it wasn't really written to satisfy, so no amount of metadata rewriting recovers clicks.

#2  content_fa17add7836d36c3  score=0.959  tier=1-3  ctr_gap=0.00280  is_declining=1
  Action:          review_metadata
  Reason code:     weak_ctr_visible_page
  Confidence:      HIGH — well above median traffic — CTR gap is measured against a lot of real data
  Would be wrong:  wrong if the CTR gap reflects a genuine search-intent mismatch rather than a fixable title/meta problem — e.g. the page ranks well for a query it wasn't really written to satisfy, so no amou

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Which picks look wrong, and why?**

1 and 9 are your two false positives (is_declining = 0), and they share a specific pattern worth naming. #1 sits at position 2.42 with real traffic (25,941 impressions) and a near-zero CTR gap; #9 sits at position 57.52 with modest traffic (6,431 impressions). Both were flagged purely because their CTR is essentially zero relative to their tier's expected value - but zero-decline outcomes mean the low CTR didn't actually precede a real drop. The likely explanation: these are pages that were already weak (chronically low CTR) rather than newly weakening — the rule can't currently tell the difference between "this page has always underperformed but is stable" and "this page is about to decline," since it only looks at the current snapshot, not the trend.

**Confirming no product flags or future windows leaked in**

Every column feeding rule_score (ctr_gap, expected_ctr_for_tier, total_impressions_30d) is built purely from data at or before the decision point (2026-03-31) — none reference April or any later window. is_declining appears in the output only for review, never inside the scoring formula — visible directly in this run, where the two false positives (#1, #9) were still ranked at the top purely by rule_score, proving the label had no influence on placement. No FlyRank product flag (health_score, priority_score, action_type) appears in any of the 13 columns.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.